In [ ]:
# ============================================================
# Property Sales - Clean Observation Layer
# ============================================================
#
# Input:
#     Raw.property_sales_data
#
# Output:
#     Cleaned.property_sales_data
#
# Input grain:
#     One row per listing scrape observation.
#
# Output grain:
#     One cleaned row per Raw ObservationKey.
#
# Responsibilities:
#     1. Preserve observation identity and source provenance.
#     2. Clean malformed text without losing original source values.
#     3. Repair the known CSV column-shift pattern.
#     4. Parse listing IDs, prices, addresses, dates and property fields.
#     5. Classify data quality without deleting questionable records.
#     6. Preserve repeated observations of the same listing.
#
# This notebook does not:
#     - Select one preferred row per ListingId.
#     - Deduplicate listings across scrape dates.
#     - Resolve physical property identity.
#     - Apply dimensional modelling.
#
# Those operations belong in Curated and Gold.
# ============================================================

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DateType,
    DecimalType,
    IntegerType,
    LongType,
    StringType,
    TimestampType,
)

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

RAW_TABLE = "Raw.property_sales_data"

CLEAN_SCHEMA = "Cleaned"
CLEAN_TABLE_NAME = "property_sales_data"
CLEAN_TABLE = f"{CLEAN_SCHEMA}.{CLEAN_TABLE_NAME}"

# Broad plausibility thresholds.
#
# These values classify records; they do not remove them.
MIN_EXPECTED_SOLD_DATE = "1990-01-01"

MIN_PLAUSIBLE_PRICE = 10_000
MAX_PLAUSIBLE_PRICE = 100_000_000

MIN_PLAUSIBLE_LAND_SIZE_SQM = 20
MAX_PLAUSIBLE_LAND_SIZE_SQM = 10_000_000

MAX_PLAUSIBLE_BEDROOMS = 20
MAX_PLAUSIBLE_BATHROOMS = 20
MAX_PLAUSIBLE_CAR_SPACES = 50

REA_BASE_URL = "https://www.realestate.com.au"

NULL_LITERALS = [
    "",
    "-",
    "--",
    "null",
    "none",
    "nan",
    "n/a",
    "na",
    "undefined",
    "unknown",
    "not available",
]

In [ ]:
# ============================================================
# 3. Spark configuration
# ============================================================

# Invalid casts should become NULL rather than terminate the notebook.
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CLEAN_SCHEMA}")

In [ ]:
# ============================================================
# 4. Helper functions
# ============================================================

def clean_string_expr(column_expression):
    """
    Defensively clean a Spark string expression.

    The function:
      - normalises smart quotes and non-breaking spaces
      - removes control characters
      - removes surrounding quotes
      - replaces repeated internal whitespace
      - converts null-like literals to NULL

    It does not alter semantic content beyond basic source repair.
    """
    value = column_expression.cast("string")

    value = F.regexp_replace(value, "\u00A0", " ")
    value = F.regexp_replace(value, "[“”]", '"')
    value = F.regexp_replace(value, "[‘’]", "'")
    value = F.trim(value)
    value = F.regexp_replace(value, r"""^['"]+|['"]+$""", "")
    value = F.regexp_replace(value, r'""', '"')
    value = F.regexp_replace(value, r"[\x00-\x1F\x7F]+", " ")
    value = F.regexp_replace(value, r"\s+", " ")
    value = F.trim(value)

    return (
        F.when(
            value.isNull()
            | (value == "")
            | F.lower(value).isin(*NULL_LITERALS),
            F.lit(None).cast(StringType()),
        )
        .otherwise(value)
    )


def clean_string(column_name):
    """
    Clean a named DataFrame column.
    """
    return clean_string_expr(F.col(column_name))


def first_integer_expr(column_expression):
    """
    Extract the first signed integer-like value from an expression.

    Examples:
        "$1,250,000" -> 1250000
        "450 m²"     -> 450
        "3 cars"     -> 3
    """
    value = clean_string_expr(column_expression)
    value = F.regexp_replace(value, ",", "")

    extracted = F.regexp_extract(value, r"(-?\d+)", 1)

    return (
        F.when(
            F.length(extracted) > 0,
            extracted.cast(LongType()),
        )
        .otherwise(F.lit(None).cast(LongType()))
    )


def first_integer(column_name):
    """
    Extract the first integer-like value from a named column.
    """
    return first_integer_expr(F.col(column_name))


def bounded_integer(column_expression, minimum, maximum):
    """
    Return an INTEGER only when it falls within the supplied bounds.
    """
    value = column_expression.cast(LongType())

    return (
        F.when(
            value.between(minimum, maximum),
            value.cast(IntegerType()),
        )
        .otherwise(F.lit(None).cast(IntegerType()))
    )


def regex_long(column_expression, pattern):
    """
    Extract a regex capture group and return it as BIGINT.
    """
    extracted = F.regexp_extract(
        column_expression.cast(StringType()),
        pattern,
        1,
    )

    return (
        F.when(
            F.length(extracted) > 0,
            extracted.cast(LongType()),
        )
        .otherwise(F.lit(None).cast(LongType()))
    )


def parse_timestamp(column_expression):
    """
    Parse known scraper timestamp formats.

    The Raw value is retained separately.
    """
    value = clean_string_expr(column_expression)

    return F.coalesce(
        value.cast(TimestampType()),
        F.to_timestamp(value, "yyyy-MM-dd'T'HH:mm:ss.SSSX"),
        F.to_timestamp(value, "yyyy-MM-dd'T'HH:mm:ssX"),
        F.to_timestamp(value, "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(value, "yyyy-MM-dd HH:mm:ss.SSS"),
    )


def parse_sold_date(iso_expression, text_expression):
    """
    Parse sold dates from both the ISO field and display text field.
    """
    iso_value = clean_string_expr(iso_expression)
    text_value = clean_string_expr(text_expression)

    return F.coalesce(
        iso_expression.cast(DateType()),
        F.to_date(iso_value, "yyyy-MM-dd"),
        F.to_date(iso_value, "d/MM/yyyy"),
        F.to_date(iso_value, "dd/MM/yyyy"),
        F.to_date(text_value, "yyyy-MM-dd"),
        F.to_date(text_value, "d/MM/yyyy"),
        F.to_date(text_value, "dd/MM/yyyy"),
        F.to_date(text_value, "d MMM yyyy"),
        F.to_date(text_value, "dd MMM yyyy"),
        F.to_date(text_value, "d MMMM yyyy"),
        F.to_date(text_value, "dd MMMM yyyy"),
    )


def table_exists(table_name):
    """
    Return True if a Spark catalog table exists.
    """
    return spark.catalog.tableExists(table_name)

In [ ]:
# ============================================================
# 5. Validate Raw input
# ============================================================

if not table_exists(RAW_TABLE):
    raise RuntimeError(
        f"Required Raw table does not exist: {RAW_TABLE}"
    )

raw = spark.table(RAW_TABLE)

required_raw_columns = [
    "ObservationKey",
    "RecordHash",
    "sourceUrl",
    "pageNumber",
    "ordinalOnPage",
    "price",
    "priceValue",
    "address",
    "detailPath",
    "detailUrl",
    "bedrooms",
    "bathrooms",
    "carSpaces",
    "landSize",
    "landSizeSqm",
    "propertyType",
    "soldDate",
    "soldDateISO",
    "scrapedAt",
    "SourceFile",
    "LoadedAt",
    "LoadDate",
]

missing_raw_columns = [
    column_name
    for column_name in required_raw_columns
    if column_name not in raw.columns
]

if missing_raw_columns:
    raise RuntimeError(
        "Raw table is missing required columns: "
        + ", ".join(missing_raw_columns)
    )


raw_summary = raw.select(
    F.count("*").alias("RawRowCount"),
    F.countDistinct("ObservationKey").alias(
        "DistinctObservationCount"
    ),
    F.sum(
        F.when(
            F.col("ObservationKey").isNull(),
            1,
        ).otherwise(0)
    ).alias("NullObservationKeyCount"),
).first()

raw_count = raw_summary["RawRowCount"]

print(f"Loaded {raw_count:,} rows from {RAW_TABLE}")

if raw_summary["NullObservationKeyCount"] != 0:
    raise RuntimeError(
        "Raw table contains NULL ObservationKey values."
    )

if (
    raw_summary["RawRowCount"]
    != raw_summary["DistinctObservationCount"]
):
    raise RuntimeError(
        "Raw table contains duplicate ObservationKey values."
    )

In [ ]:
# ============================================================
# 6. Preserve original source values
# ============================================================

# Original values remain available so parser and repair logic can be
# revised later without returning to the CSV archive.
staged = (
    raw

    # Observation-level identity
    .withColumn(
        "ObservationKey",
        clean_string("ObservationKey"),
    )
    .withColumn(
        "RecordHash",
        clean_string("RecordHash"),
    )

    # Source provenance
    .withColumn(
        "SourceFile",
        clean_string("SourceFile"),
    )
    .withColumn(
        "SourceUrlRaw",
        clean_string("sourceUrl"),
    )
    .withColumn(
        "SourcePageRaw",
        F.col("pageNumber").cast(StringType()),
    )
    .withColumn(
        "SourceOrdinalRaw",
        F.col("ordinalOnPage").cast(StringType()),
    )

    # Original listing source values
    .withColumn(
        "PriceTextRaw",
        clean_string("price"),
    )
    .withColumn(
        "PriceValueSourceRaw",
        F.col("priceValue").cast(StringType()),
    )
    .withColumn(
        "AddressTextRaw",
        clean_string("address"),
    )
    .withColumn(
        "DetailPathRaw",
        clean_string("detailPath"),
    )
    .withColumn(
        "DetailUrlRaw",
        clean_string("detailUrl"),
    )
    .withColumn(
        "BedroomsSourceRaw",
        F.col("bedrooms").cast(StringType()),
    )
    .withColumn(
        "BathroomsSourceRaw",
        F.col("bathrooms").cast(StringType()),
    )
    .withColumn(
        "CarSpacesSourceRaw",
        F.col("carSpaces").cast(StringType()),
    )
    .withColumn(
        "LandSizeTextRaw",
        clean_string("landSize"),
    )
    .withColumn(
        "LandSizeSqmSourceRaw",
        F.col("landSizeSqm").cast(StringType()),
    )
    .withColumn(
        "PropertyTypeRaw",
        clean_string("propertyType"),
    )
    .withColumn(
        "SoldDateTextRaw",
        clean_string("soldDate"),
    )
    .withColumn(
        "SoldDateISORaw",
        F.col("soldDateISO").cast(StringType()),
    )
    .withColumn(
        "ScrapedAtRaw",
        clean_string("scrapedAt"),
    )

    # Raw load metadata
    .withColumn(
        "RawLoadedAt",
        F.col("LoadedAt").cast(TimestampType()),
    )
    .withColumn(
        "RawLoadDate",
        F.col("LoadDate").cast(DateType()),
    )
)

In [ ]:
# ============================================================
# 7. Reparse source values defensively
# ============================================================

# Although the Raw notebook applies minimal typing, this notebook
# reparses from retained values so the Clean layer remains robust to
# historical Raw schema differences.
staged = (
    staged
    .withColumn(
        "SourcePage",
        bounded_integer(
            first_integer_expr(F.col("SourcePageRaw")),
            0,
            100_000,
        ),
    )
    .withColumn(
        "SourceOrdinal",
        bounded_integer(
            first_integer_expr(F.col("SourceOrdinalRaw")),
            0,
            100_000,
        ),
    )
    .withColumn(
        "PriceValueRaw",
        first_integer_expr(F.col("PriceValueSourceRaw")),
    )
    .withColumn(
        "BedroomsRaw",
        first_integer_expr(F.col("BedroomsSourceRaw")),
    )
    .withColumn(
        "BathroomsRaw",
        first_integer_expr(F.col("BathroomsSourceRaw")),
    )
    .withColumn(
        "CarSpacesRaw",
        first_integer_expr(F.col("CarSpacesSourceRaw")),
    )
    .withColumn(
        "LandSizeSqmRaw",
        first_integer_expr(F.col("LandSizeSqmSourceRaw")),
    )
)

In [ ]:
# ============================================================
# 8. Detect and repair shifted CSV columns
# ============================================================

# Known malformed pattern:
#
#     address contains a comma
#     -> trailing address portion lands in detailPath
#     -> actual detailPath lands in detailUrl
#     -> subsequent property count columns shift to the right
#
# Detection intentionally requires:
#     - DetailPathRaw is not already a /sold/ path
#     - DetailUrlRaw is a /sold/ path
#
# The source record is retained and marked rather than silently altered.
staged = (
    staged
    .withColumn(
        "InputWasColumnShifted",
        (
            F.col("DetailPathRaw").isNotNull()
            & ~F.col("DetailPathRaw").rlike(r"^/sold/")
            & F.col("DetailUrlRaw").rlike(r"^/sold/")
        ).cast(BooleanType()),
    )
    .withColumn(
        "AddressText",
        F.when(
            F.col("InputWasColumnShifted"),
            F.concat_ws(
                ", ",
                F.col("AddressTextRaw"),
                F.col("DetailPathRaw"),
            ),
        ).otherwise(F.col("AddressTextRaw")),
    )
    .withColumn(
        "DetailPath",
        F.when(
            F.col("InputWasColumnShifted"),
            F.col("DetailUrlRaw"),
        ).otherwise(F.col("DetailPathRaw")),
    )
    .withColumn(
        "BedroomsCandidate",
        F.when(
            F.col("InputWasColumnShifted"),
            F.col("BathroomsRaw"),
        ).otherwise(F.col("BedroomsRaw")),
    )
    .withColumn(
        "BathroomsCandidate",
        F.when(
            F.col("InputWasColumnShifted"),
            F.col("CarSpacesRaw"),
        ).otherwise(F.col("BathroomsRaw")),
    )
    .withColumn(
        "CarSpacesCandidate",
        F.when(
            F.col("InputWasColumnShifted"),
            first_integer_expr(F.col("LandSizeTextRaw")),
        ).otherwise(F.col("CarSpacesRaw")),
    )
)

In [ ]:
# ============================================================
# 9. Standardise paths and URLs
# ============================================================

staged = (
    staged
    .withColumn(
        "DetailPath",
        F.when(
            F.col("DetailPath").rlike(
                r"^https?://(?:www\.)?realestate\.com\.au"
            ),
            F.regexp_extract(
                F.col("DetailPath"),
                r"^https?://(?:www\.)?realestate\.com\.au([^?#]+)",
                1,
            ),
        )
        .when(
            F.col("DetailPath").startswith("/"),
            F.col("DetailPath"),
        )
        .otherwise(F.lit(None).cast(StringType())),
    )
    .withColumn(
        "DetailUrl",
        F.when(
            F.col("DetailUrlRaw").rlike(r"^https?://"),
            F.regexp_replace(
                F.col("DetailUrlRaw"),
                r"^http://",
                "https://",
            ),
        )
        .when(
            F.col("DetailPath").startswith("/"),
            F.concat(
                F.lit(REA_BASE_URL),
                F.col("DetailPath"),
            ),
        )
        .otherwise(F.lit(None).cast(StringType())),
    )
)

In [ ]:
# ============================================================
# 10. Parse identifiers and source metadata
# ============================================================

clean = (
    staged
    .withColumn(
        "ListingId",
        F.coalesce(
            regex_long(
                F.col("DetailPath"),
                r"(\d+)(?:[/?#].*)?$",
            ),
            regex_long(
                F.col("DetailUrl"),
                r"(\d+)(?:[/?#].*)?$",
            ),
        ),
    )
    .withColumn(
        "SourceUrl",
        F.col("SourceUrlRaw"),
    )
)

In [ ]:
# ============================================================
# 11. Parse and classify addresses
# ============================================================

clean = (
    clean
    .withColumn(
        "IsAddressSuppressed",
        F.coalesce(
            F.lower(F.col("AddressText")).rlike(
                r"^(address\s+(available|withheld)|"
                r"address\s+upon\s+request|"
                r"contact\s+agent\s+for\s+address)"
            ),
            F.lit(False),
        ).cast(BooleanType()),
    )
    .withColumn(
        "FullAddress",
        F.when(
            F.col("IsAddressSuppressed"),
            F.lit(None).cast(StringType()),
        ).otherwise(F.col("AddressText")),
    )
    .withColumn(
        "Suburb",
        F.when(
            F.col("FullAddress").contains(","),
            F.trim(
                F.regexp_extract(
                    F.col("FullAddress"),
                    r",\s*([^,]+?)\s*$",
                    1,
                )
            ),
        ).otherwise(F.lit(None).cast(StringType())),
    )
    .withColumn(
        "Suburb",
        F.regexp_replace(
            F.col("Suburb"),
            r"\s+(WA|Western Australia)\s+\d{4}$",
            "",
        ),
    )
    .withColumn(
        "Suburb",
        F.regexp_replace(
            F.col("Suburb"),
            r"\s+\d{4}$",
            "",
        ),
    )
    .withColumn(
        "Suburb",
        F.when(
            F.length(F.trim(F.col("Suburb"))) > 0,
            F.initcap(F.trim(F.col("Suburb"))),
        ).otherwise(F.lit(None).cast(StringType())),
    )
    .withColumn(
        "Postcode",
        F.when(
            F.col("FullAddress").isNotNull(),
            regex_long(
                F.col("FullAddress"),
                r"\b((?:6)\d{3})\b",
            ).cast(IntegerType()),
        ).otherwise(F.lit(None).cast(IntegerType())),
    )
)

In [ ]:
# ============================================================
# 12. Parse property attributes
# ============================================================

clean = (
    clean
    .withColumn(
        "Bedrooms",
        bounded_integer(
            F.col("BedroomsCandidate"),
            0,
            MAX_PLAUSIBLE_BEDROOMS,
        ),
    )
    .withColumn(
        "Bathrooms",
        bounded_integer(
            F.col("BathroomsCandidate"),
            0,
            MAX_PLAUSIBLE_BATHROOMS,
        ),
    )
    .withColumn(
        "CarSpaces",
        bounded_integer(
            F.col("CarSpacesCandidate"),
            0,
            MAX_PLAUSIBLE_CAR_SPACES,
        ),
    )
    .withColumn(
        "LandSizeSqm",
        F.coalesce(
            F.when(
                F.col("LandSizeSqmRaw") > 0,
                F.col("LandSizeSqmRaw"),
            ),
            F.when(
                first_integer_expr(F.col("LandSizeTextRaw")) > 0,
                first_integer_expr(
                    F.col("LandSizeTextRaw")
                ),
            ),
        ).cast(LongType()),
    )
    .withColumn(
        "PropertyType",
        F.when(
            F.col("PropertyTypeRaw").isNotNull(),
            F.initcap(F.col("PropertyTypeRaw")),
        ).otherwise(F.lit(None).cast(StringType())),
    )
)

In [ ]:
# ============================================================
# 13. Parse prices
# ============================================================

price_text_lower = F.lower(
    F.coalesce(
        F.col("PriceTextRaw"),
        F.lit(""),
    )
)

clean = (
    clean
    .withColumn(
        "PriceValue",
        F.when(
            F.col("PriceValueRaw") > 0,
            F.col("PriceValueRaw"),
        ).otherwise(F.lit(None).cast(LongType())),
    )
    .withColumn(
        "PriceText",
        F.col("PriceTextRaw"),
    )
    .withColumn(
        "IsContactAgent",
        price_text_lower.rlike(
            r"(contact|call)\s+(agent|for price)"
            r"|price\s+on\s+application"
            r"|\bpoa\b"
        ).cast(BooleanType()),
    )
    .withColumn(
        "IsPriceRange",
        price_text_lower.rlike(
            r"\$?\s*[\d,.]+\s*[-–—]\s*\$?\s*[\d,.]+"
            r"|price\s+range"
            r"|\brange\b"
            r"|between\s+\$?"
        ).cast(BooleanType()),
    )
    .withColumn(
        "IsPriceFrom",
        price_text_lower.rlike(
            r"^(from|offers?\s+(from|over|above)|"
            r"starting\s+(from|at)|"
            r"mid|high|low)"
        ).cast(BooleanType()),
    )
    .withColumn(
        "IsPriceWithheld",
        price_text_lower.rlike(
            r"withheld|not\s+disclosed|undisclosed"
        ).cast(BooleanType()),
    )
    .withColumn(
        "PriceType",
        F.when(
            F.col("IsContactAgent"),
            F.lit("CONTACT"),
        )
        .when(
            F.col("IsPriceWithheld"),
            F.lit("WITHHELD"),
        )
        .when(
            F.col("IsPriceRange"),
            F.lit("RANGE"),
        )
        .when(
            F.col("IsPriceFrom"),
            F.lit("INDICATIVE"),
        )
        .when(
            F.col("PriceValue").isNotNull(),
            F.lit("NUMERIC"),
        )
        .when(
            F.col("PriceText").isNotNull(),
            F.lit("UNPARSED"),
        )
        .otherwise(F.lit("MISSING")),
    )
    .withColumn(
        "HasNumericPrice",
        F.col("PriceValue").isNotNull(),
    )
)

In [ ]:
# ============================================================
# 14. Parse dates and scrape metadata
# ============================================================

clean = (
    clean
    .withColumn(
        "SoldDate",
        parse_sold_date(
            F.col("soldDateISO"),
            F.col("SoldDateTextRaw"),
        ),
    )
    .withColumn(
        "ScrapedAt",
        parse_timestamp(F.col("scrapedAt")),
    )
    .withColumn(
        "ScrapeDate",
        F.to_date(F.col("ScrapedAt")),
    )
    .withColumn(
        "ScrapeYear",
        F.year(F.col("ScrapeDate")).cast(IntegerType()),
    )
    .withColumn(
        "ScrapeMonth",
        F.month(F.col("ScrapeDate")).cast(IntegerType()),
    )
)

In [ ]:
# ============================================================
# 15. Add field-level quality classifications
# ============================================================

clean = (
    clean

    # Listing identifier
    .withColumn(
        "ListingIdQuality",
        F.when(
            F.col("ListingId").isNull(),
            F.lit("MISSING"),
        ).otherwise(F.lit("VALID")),
    )

    # Sold date
    .withColumn(
        "SoldDateQuality",
        F.when(
            F.col("SoldDate").isNull(),
            F.lit("MISSING"),
        )
        .when(
            F.col("ScrapeDate").isNotNull()
            & (F.col("SoldDate") > F.col("ScrapeDate")),
            F.lit("AFTER_SCRAPE_DATE"),
        )
        .when(
            F.col("SoldDate") < F.lit(
                MIN_EXPECTED_SOLD_DATE
            ).cast(DateType()),
            F.lit("HISTORICAL_OUTLIER"),
        )
        .otherwise(F.lit("VALID")),
    )

    # Price
    .withColumn(
        "PriceQuality",
        F.when(
            F.col("PriceValue").isNull()
            & F.col("IsContactAgent"),
            F.lit("CONTACT_AGENT"),
        )
        .when(
            F.col("PriceValue").isNull()
            & F.col("IsPriceWithheld"),
            F.lit("WITHHELD"),
        )
        .when(
            F.col("PriceValue").isNull()
            & F.col("IsPriceRange"),
            F.lit("RANGE_WITHOUT_NUMERIC_VALUE"),
        )
        .when(
            F.col("PriceValue").isNull(),
            F.lit("MISSING_OR_UNPARSED"),
        )
        .when(
            F.col("PriceValue") < MIN_PLAUSIBLE_PRICE,
            F.lit("IMPLAUSIBLY_LOW"),
        )
        .when(
            F.col("PriceValue") > MAX_PLAUSIBLE_PRICE,
            F.lit("IMPLAUSIBLY_HIGH"),
        )
        .when(
            F.col("IsPriceRange"),
            F.lit("RANGE_NUMERIC_VALUE"),
        )
        .when(
            F.col("IsPriceFrom"),
            F.lit("INDICATIVE_NUMERIC_VALUE"),
        )
        .otherwise(F.lit("VALID_NUMERIC")),
    )

    # Address
    .withColumn(
        "AddressQuality",
        F.when(
            F.col("IsAddressSuppressed"),
            F.lit("SUPPRESSED"),
        )
        .when(
            F.col("FullAddress").isNull(),
            F.lit("MISSING"),
        )
        .when(
            F.col("Suburb").isNull(),
            F.lit("UNPARSED_SUBURB"),
        )
        .otherwise(F.lit("VALID")),
    )

    # Land size
    .withColumn(
        "LandSizeQuality",
        F.when(
            F.col("LandSizeSqm").isNull(),
            F.lit("MISSING"),
        )
        .when(
            F.col("LandSizeSqm") <= 0,
            F.lit("INVALID"),
        )
        .when(
            F.col("LandSizeSqm")
            < MIN_PLAUSIBLE_LAND_SIZE_SQM,
            F.lit("IMPLAUSIBLY_SMALL"),
        )
        .when(
            F.col("LandSizeSqm")
            > MAX_PLAUSIBLE_LAND_SIZE_SQM,
            F.lit("IMPLAUSIBLY_LARGE"),
        )
        .otherwise(F.lit("VALID")),
    )

    # Scrape timestamp
    .withColumn(
        "ScrapeTimestampQuality",
        F.when(
            F.col("ScrapedAt").isNull(),
            F.lit("MISSING_OR_UNPARSED"),
        )
        .when(
            F.col("ScrapedAt")
            > F.current_timestamp() + F.expr("INTERVAL 1 DAY"),
            F.lit("FUTURE"),
        )
        .otherwise(F.lit("VALID")),
    )
)

In [ ]:
# ============================================================
# 16. Add quality flags and derived metrics
# ============================================================

clean = (
    clean
    .withColumn(
        "HasValidListingId",
        F.col("ListingIdQuality") == "VALID",
    )
    .withColumn(
        "HasValidSoldDate",
        F.col("SoldDateQuality") == "VALID",
    )
    .withColumn(
        "HasValidNumericPrice",
        F.col("PriceQuality") == "VALID_NUMERIC",
    )
    .withColumn(
        "HasUsableAddress",
        F.col("AddressQuality") == "VALID",
    )
    .withColumn(
        "HasValidLandSize",
        F.col("LandSizeQuality") == "VALID",
    )
    .withColumn(
        "PricePerSqm",
        F.when(
            F.col("PriceValue").isNotNull()
            & (F.col("PriceValue") > 0)
            & (F.col("LandSizeSqm").isNotNull())
            & (F.col("LandSizeSqm") > 0),
            (
                F.col("PriceValue").cast(DecimalType(20, 2))
                / F.col("LandSizeSqm")
            ).cast(DecimalType(20, 2)),
        ).otherwise(F.lit(None).cast(DecimalType(20, 2))),
    )
    .withColumn(
        "CleanedAt",
        F.current_timestamp(),
    )
)

In [ ]:
# ============================================================
# 17. Select final Cleaned schema
# ============================================================

clean = clean.select(
    # Observation identity
    "ObservationKey",
    "RecordHash",
    "ListingId",

    # Source provenance
    "SourceFile",
    "SourceUrl",
    "SourcePage",
    "SourceOrdinal",
    "RawLoadedAt",
    "RawLoadDate",
    "CleanedAt",

    # Original source values
    "PriceTextRaw",
    "PriceValueSourceRaw",
    "AddressTextRaw",
    "DetailPathRaw",
    "DetailUrlRaw",
    "BedroomsSourceRaw",
    "BathroomsSourceRaw",
    "CarSpacesSourceRaw",
    "LandSizeTextRaw",
    "LandSizeSqmSourceRaw",
    "PropertyTypeRaw",
    "SoldDateTextRaw",
    "SoldDateISORaw",
    "ScrapedAtRaw",

    # Parsed listing identity and URLs
    "DetailPath",
    "DetailUrl",

    # Address and geography
    "FullAddress",
    "Suburb",
    "Postcode",
    "IsAddressSuppressed",

    # Property attributes
    "PropertyType",
    "Bedrooms",
    "Bathrooms",
    "CarSpaces",
    "LandSizeSqm",

    # Price attributes
    "PriceText",
    "PriceValue",
    "PriceType",
    "HasNumericPrice",
    "IsContactAgent",
    "IsPriceRange",
    "IsPriceFrom",
    "IsPriceWithheld",
    "PricePerSqm",

    # Dates
    "SoldDate",
    "ScrapedAt",
    "ScrapeDate",
    "ScrapeYear",
    "ScrapeMonth",

    # Repair metadata
    "InputWasColumnShifted",

    # Quality classifications
    "ListingIdQuality",
    "SoldDateQuality",
    "PriceQuality",
    "AddressQuality",
    "LandSizeQuality",
    "ScrapeTimestampQuality",

    # Convenience quality flags
    "HasValidListingId",
    "HasValidSoldDate",
    "HasValidNumericPrice",
    "HasUsableAddress",
    "HasValidLandSize",
)

In [ ]:
# ============================================================
# 18. Enforce observation-level uniqueness
# ============================================================

clean_summary_before_write = clean.select(
    F.count("*").alias("CleanRowCount"),
    F.countDistinct("ObservationKey").alias(
        "DistinctObservationCount"
    ),
    F.sum(
        F.when(
            F.col("ObservationKey").isNull(),
            1,
        ).otherwise(0)
    ).alias("NullObservationKeyCount"),
).first()

if clean_summary_before_write["NullObservationKeyCount"] != 0:
    raise RuntimeError(
        "Clean transformation produced NULL ObservationKey values."
    )

if (
    clean_summary_before_write["CleanRowCount"]
    != clean_summary_before_write["DistinctObservationCount"]
):
    raise RuntimeError(
        "Clean transformation produced duplicate ObservationKey values."
    )

if clean_summary_before_write["CleanRowCount"] != raw_count:
    raise RuntimeError(
        "Clean row count does not match Raw row count. "
        f"Raw={raw_count:,}; "
        f"Clean={clean_summary_before_write['CleanRowCount']:,}"
    )

print(
    "Prepared "
    f"{clean_summary_before_write['CleanRowCount']:,} "
    "cleaned observations."
)

In [ ]:
# ============================================================
# 19. Write Cleaned Delta table
# ============================================================

(
    clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CLEAN_TABLE)
)

print(f"Wrote cleaned Delta table: {CLEAN_TABLE}")

In [ ]:
# ============================================================
# 20. Validate written table
# ============================================================

summary = spark.sql(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT ObservationKey)
            AS distinct_observation_count,
        COUNT(DISTINCT ListingId)
            AS distinct_listing_count,

        SUM(
            CASE
                WHEN ObservationKey IS NULL THEN 1
                ELSE 0
            END
        ) AS null_observation_key_count,

        SUM(
            CASE
                WHEN ListingId IS NULL THEN 1
                ELSE 0
            END
        ) AS null_listing_id_count,

        SUM(
            CASE
                WHEN PriceValue IS NULL THEN 1
                ELSE 0
            END
        ) AS null_price_count,

        SUM(
            CASE
                WHEN FullAddress IS NULL THEN 1
                ELSE 0
            END
        ) AS null_address_count,

        SUM(
            CASE
                WHEN InputWasColumnShifted THEN 1
                ELSE 0
            END
        ) AS repaired_column_shift_count,

        MIN(SoldDate) AS min_sold_date,
        MAX(SoldDate) AS max_sold_date,
        MIN(ScrapedAt) AS min_scraped_at,
        MAX(ScrapedAt) AS max_scraped_at,
        MIN(RawLoadedAt) AS min_raw_loaded_at,
        MAX(RawLoadedAt) AS max_raw_loaded_at

    FROM {CLEAN_TABLE}
    """
)

summary.show(truncate=False)

written_summary = summary.first()

if (
    written_summary["row_count"]
    != written_summary["distinct_observation_count"]
):
    raise RuntimeError(
        "Written Cleaned table does not contain one row per "
        "ObservationKey."
    )

if written_summary["row_count"] != raw_count:
    raise RuntimeError(
        "Written Cleaned row count does not match Raw row count."
    )

In [ ]:
# ============================================================
# 21. Profile quality classifications
# ============================================================

quality_dimensions = [
    "ListingIdQuality",
    "SoldDateQuality",
    "PriceQuality",
    "AddressQuality",
    "LandSizeQuality",
    "ScrapeTimestampQuality",
]

for quality_column in quality_dimensions:
    print(f"\n{quality_column}:")

    spark.sql(
        f"""
        SELECT
            {quality_column},
            COUNT(*) AS RowCount,
            ROUND(
                COUNT(*) * 100.0
                / SUM(COUNT(*)) OVER (),
                2
            ) AS Percentage

        FROM {CLEAN_TABLE}

        GROUP BY
            {quality_column}

        ORDER BY
            RowCount DESC,
            {quality_column}
        """
    ).show(truncate=False)

In [ ]:
# ============================================================
# 22. Profile repeated listing observations
# ============================================================

print("\nRepeated listing observations:")

spark.sql(
    f"""
    SELECT
        ListingId,
        COUNT(*) AS ObservationCount,
        COUNT(DISTINCT RecordHash) AS DistinctRecordVersions,
        MIN(ScrapedAt) AS FirstScrapedAt,
        MAX(ScrapedAt) AS LastScrapedAt,
        COUNT(DISTINCT PriceValue) AS DistinctNumericPrices,
        COUNT(DISTINCT FullAddress) AS DistinctAddresses

    FROM {CLEAN_TABLE}

    WHERE ListingId IS NOT NULL

    GROUP BY ListingId

    HAVING COUNT(*) > 1

    ORDER BY
        ObservationCount DESC,
        ListingId

    LIMIT 50
    """
).show(truncate=False)

In [ ]:
# ============================================================
# 23. Inspect repaired or questionable records
# ============================================================

print("\nRepaired and questionable records:")

spark.sql(
    f"""
    SELECT
        ObservationKey,
        ListingId,
        PriceText,
        PriceValue,
        PriceQuality,
        FullAddress,
        Suburb,
        Postcode,
        AddressQuality,
        DetailPath,
        DetailUrl,
        Bedrooms,
        Bathrooms,
        CarSpaces,
        LandSizeSqm,
        LandSizeQuality,
        SoldDate,
        SoldDateQuality,
        ScrapedAt,
        ScrapeTimestampQuality,
        InputWasColumnShifted,
        SourceFile,
        SourcePage,
        SourceOrdinal

    FROM {CLEAN_TABLE}

    WHERE
        ListingIdQuality <> 'VALID'
        OR SoldDateQuality <> 'VALID'
        OR PriceQuality NOT IN (
            'VALID_NUMERIC',
            'CONTACT_AGENT',
            'WITHHELD'
        )
        OR AddressQuality <> 'VALID'
        OR LandSizeQuality <> 'VALID'
        OR ScrapeTimestampQuality <> 'VALID'
        OR InputWasColumnShifted = true

    ORDER BY
        InputWasColumnShifted DESC,
        ListingId,
        ScrapedAt

    LIMIT 100
    """
).show(truncate=False)

In [ ]:
# ============================================================
# 24. Final status
# ============================================================

print(
    f"""
Clean property-sales transformation complete.

Source table:                  {RAW_TABLE}
Target table:                  {CLEAN_TABLE}
Raw observations:              {raw_count:,}
Clean observations:            {written_summary["row_count"]:,}
Distinct observations:         {written_summary["distinct_observation_count"]:,}
Distinct listings:             {written_summary["distinct_listing_count"]:,}
Missing listing IDs:           {written_summary["null_listing_id_count"]:,}
Missing numeric prices:        {written_summary["null_price_count"]:,}
Missing usable addresses:      {written_summary["null_address_count"]:,}
Repaired shifted records:      {written_summary["repaired_column_shift_count"]:,}
Minimum sold date:             {written_summary["min_sold_date"]}
Maximum sold date:             {written_summary["max_sold_date"]}
Minimum scrape timestamp:      {written_summary["min_scraped_at"]}
Maximum scrape timestamp:      {written_summary["max_scraped_at"]}
"""
)